<a href="https://colab.research.google.com/github/FatikImran/vyrothon-stage-1/blob/main/vyrothon_stage_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: clone repo and install dependencies

!git clone https://github.com/FatikImran/vyrothon-stage-1
%cd vyrothon-stage-1
!python -m pip install --upgrade pip
!pip install -r requirements.txt

Cloning into 'vyrothon-stage-1'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 32 (delta 6), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 18.40 KiB | 319.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/vyrothon-stage-1/vyrothon-stage-1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 68.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 38.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 81.0 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled dat

In [6]:
# Cell 2: quick sanity check

!python -m unittest discover -s tests -p "test_*.py"
!python -m pocket_agent.eval --path starter/shadow_eval.jsonl

......
----------------------------------------------------------------------
Ran 6 tests in 6.520s

OK
{
  "exact_match": 0.6111111111111112,
  "tool_match": 0.8888888888888888,
  "count": 18.0
}


In [7]:
# Cell 3: generate synthetic training data
!python -m pocket_agent.data --output data/generated/train.jsonl --count 2400 --seed 7

wrote 2400 examples to data/generated/train.jsonl


In [12]:
# Cell 4: train LoRA adapter
!git pull
!python -m pocket_agent.train --data data/generated/train.jsonl --output artifacts/lora --base-model Qwen/Qwen2.5-1.5B-Instruct

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 362 bytes | 362.00 KiB/s, done.
From https://github.com/FatikImran/vyrothon-stage-1
   757c7c0..8e60283  main       -> origin/main
Updating 757c7c0..8e60283
Fast-forward
 pocket_agent/train.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Loading weights: 100% 338/338 [00:01<00:00, 273.82it/s, Materializing param=model.norm.weight]
Adding EOS to train dataset: 100% 2400/2400 [00:00<00:00, 41844.01 examples/s]
Tokenizing train dataset: 100% 2400/2400 [00:00<00:00, 3329.25 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
  0%

In [13]:
# Cell 5: quantize/merge model
!python -m pocket_agent.quantize --adapter artifacts/lora --output artifacts/quantized

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:02<00:00, 124.05it/s, Materializing param=model.norm.weight]
Writing model shards: 100% 1/1 [00:26<00:00, 26.40s/it]
saved quantized/merged artifact to artifacts/quantized


In [14]:
# Cell 6: check artifacts
!du -h artifacts -d 2
!ls -R artifacts

2.9G	artifacts/quantized
24M	artifacts/lora/checkpoint-400
24M	artifacts/lora/checkpoint-500
24M	artifacts/lora/checkpoint-200
87M	artifacts/lora
3.0G	artifacts
artifacts:
lora  quantized

artifacts/lora:
adapter_config.json	   checkpoint-400	  tokenizer.json
adapter_model.safetensors  checkpoint-500	  training_args.bin
chat_template.jinja	   README.md
checkpoint-200		   tokenizer_config.json

artifacts/lora/checkpoint-200:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json

artifacts/lora/checkpoint-400:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pth	  trainer_state.json
chat_template.jinja	   scheduler.pt		  training_args.bin
optimizer.pt		   tokenizer_config.json

artifacts/lora/checkpoint-500:
adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  rng_state.pt

In [15]:
# Cell 7: smoke-test inference contract
import inference
print(inference.run("What's the weather in Paris in F?", []))
print(inference.run("Convert 10 USD to EUR", []))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<tool_call>{"tool":"weather","args":{"location":"Paris","unit":"F"}}</tool_call>
<tool_call>{"tool":"currency","args":{"amount":10.0,"from":"USD","to":"EUR"}}</tool_call>


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
